# TRIAGE-EG Kaggle bootstrap
Notebook này chỉ clone exact commit, cài package và gọi script/test trong repository. Với private repo, đọc token từ Kaggle Secrets nhưng không in hoặc commit token.

In [ ]:
import os
REPO_URL = os.environ.get("AIC_REPO_URL", "https://github.com/Irthn1311/AIC2026_TeamPTK_SGU.git")
COMMIT_SHA = os.environ.get("AIC_REPO_REF", "TRIAGEEG")
REPO_DIR = os.environ.get("AIC_REPO_DIR", "/kaggle/working/AIC2026_TeamPTK_SGU")

In [ ]:
import subprocess

subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
subprocess.run(["git", "-C", REPO_DIR, "checkout", "--detach", COMMIT_SHA], check=True)

In [ ]:
import sys

subprocess.run([sys.executable, "-m", "pip", "install", "-e", REPO_DIR], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-r", f"{REPO_DIR}/kaggle/requirements-kaggle.txt"], check=True)

In [ ]:
subprocess.run([sys.executable, "scripts/prepare_kaggle_assets.py"], cwd=REPO_DIR, check=True)

In [ ]:
print(sys.version)
commit = subprocess.run(
    ["git", "-C", REPO_DIR, "rev-parse", "HEAD"], check=True, capture_output=True, text=True
).stdout.strip()
print("git commit:", commit)

In [ ]:
import os

os.environ.setdefault("AIC_DATA_ROOT", "/kaggle/input/datasets/nadkli/dataset-aic")
os.environ.setdefault("AIC_OUTPUT_ROOT", "/kaggle/working/artifacts")
os.environ.setdefault("AIC_AUDIT_OUTPUT_ROOT", "/kaggle/working/triage_eg_stage0_audit")
os.environ.setdefault("AIC_STAGE0_ROOT", "/kaggle/working/triage_eg_stage0_audit")
os.environ.setdefault("AIC_STAGE1_OUTPUT_ROOT", "/kaggle/working/triage_eg_stage1_baseline")

In [ ]:
cmd = [
    sys.executable,
    "scripts/run_kaggle_preprocessing.py",
    "--output-root",
    os.environ["AIC_OUTPUT_ROOT"],
    "--manifest-limit",
    "20",
    "--skip-assets",
]
if os.environ.get("AIC_FULL_RUN", "0") == "1":
    cmd.append("--full")
else:
    cmd.extend(["--smoke-video-count", os.environ.get("AIC_SMOKE_VIDEO_COUNT", "1")])
subprocess.run(cmd, cwd=REPO_DIR, check=True)

In [ ]:
if os.environ.get("AIC_RUN_PYTEST", "0") == "1":
    subprocess.run([sys.executable, "-m", "pytest", "-q"], cwd=REPO_DIR, check=True)
subprocess.run([sys.executable, "scripts/kaggle_output_manifest.py", "--limit", "20", "--output-root", os.environ["AIC_OUTPUT_ROOT"]], cwd=REPO_DIR, check=True)